In [1]:
from pathlib import Path
import duckdb

PROJECT_ROOT = next(
    folder
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").exists()
)

DATA_PATH = PROJECT_ROOT / "data/raw/openspec_code_refs.parquet"

assert DATA_PATH.exists(), f"Arquivo não encontrado: {DATA_PATH}"
print(f"Projeto: {PROJECT_ROOT}")
print(f"Dados: {DATA_PATH.name} ({DATA_PATH.stat().st_size / 1_000_000:.1f} MB)")

Projeto: /Users/pedroodorize/.codex/.chatgpt-projects/g-p-6abaf5bf87e481919dc29fec1c4d1433/msr2027-specmine
Dados: openspec_code_refs.parquet (10.4 MB)


In [2]:
schema = duckdb.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{DATA_PATH.as_posix()}')"
).df()

schema

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,repo_name,VARCHAR,YES,None,None,None
2,artifact_path,VARCHAR,YES,None,None,None
3,code_path,VARCHAR,YES,None,None,None
4,task_done,BIGINT,YES,None,None,None
5,anchor_sha,VARCHAR,YES,None,None,None
6,exists_at_anchor,BIGINT,YES,None,None,None
7,resolved_path,VARCHAR,YES,None,None,None


In [3]:
quality = duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_references,
        COUNT(DISTINCT repo_name) AS n_repositories,
        COUNT(DISTINCT artifact_path) AS n_artifacts,
        COUNT(*) FILTER (WHERE id IS NULL) AS missing_id,
        COUNT(*) - COUNT(DISTINCT id) AS duplicated_ids,
        COUNT(*) FILTER (
            WHERE repo_name IS NULL
               OR artifact_path IS NULL
               OR code_path IS NULL
        ) AS missing_reference_fields,
        COUNT(*) FILTER (WHERE task_done IS NULL) AS missing_task_done,
        COUNT(*) FILTER (WHERE exists_at_anchor IS NULL) AS missing_exists_at_anchor,
        COUNT(DISTINCT task_done) AS task_done_distinct_values,
        COUNT(DISTINCT exists_at_anchor) AS exists_distinct_values
    FROM read_parquet('{DATA_PATH.as_posix()}')
    """
).df()

quality

,n_references,n_repositories,n_artifacts,missing_id,duplicated_ids,missing_reference_fields,missing_task_done,missing_exists_at_anchor,task_done_distinct_values,exists_distinct_values
0,435401,7076,61295,0,0,0,0,22209,2,3


In [4]:
resolution_by_task = duckdb.sql(
    f"""
    SELECT
        task_done,
        CASE
            WHEN exists_at_anchor = 1 THEN 'código existe'
            WHEN exists_at_anchor = 0 THEN 'código ausente'
            ELSE 'não resolvida'
        END AS resolution_state,
        COUNT(*) AS n_references,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY task_done),
            2
        ) AS pct_within_task_status
    FROM read_parquet('{DATA_PATH.as_posix()}')
    GROUP BY task_done, resolution_state
    ORDER BY
        task_done DESC,
        CASE resolution_state
            WHEN 'código existe' THEN 1
            WHEN 'código ausente' THEN 2
            ELSE 3
        END
    """
).df()

resolution_by_task

,task_done,resolution_state,n_references,pct_within_task_status
0,1,código existe,297822,87.36
1,1,código ausente,34227,10.04
2,1,não resolvida,8865,2.60
3,0,código existe,63261,66.95
4,0,código ausente,17752,18.79
5,0,não resolvida,13474,14.26


In [5]:
raw_exists_values = duckdb.sql(
    f"""
    SELECT
        exists_at_anchor,
        COUNT(*) AS n_references,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 3) AS pct_all
    FROM read_parquet('{DATA_PATH.as_posix()}')
    GROUP BY exists_at_anchor
    ORDER BY exists_at_anchor NULLS LAST
    """
).df()

raw_exists_values

,exists_at_anchor,n_references,pct_all
0,0,51979,11.938
1,1,361083,82.931
2,2,130,0.030
3,<NA>,22209,5.101


### Regra de inclusão do piloto

A análise principal inclui somente referências com `exists_at_anchor` igual a `0` ou `1`.
Referências com valor `2` ou `NULL` são consideradas indeterminadas e serão
reportadas separadamente, mas não integram a matriz de quatro estados.

In [6]:
matrix = duckdb.sql(
    f"""
    WITH eligible_refs AS (
        SELECT
            task_done,
            exists_at_anchor
        FROM read_parquet('{DATA_PATH.as_posix()}')
        WHERE exists_at_anchor IN (0, 1)
    ),
    states AS (
        SELECT
            task_done,
            CASE
                WHEN task_done = 1 AND exists_at_anchor = 1
                    THEN 'concluída + código existe'
                WHEN task_done = 1 AND exists_at_anchor = 0
                    THEN 'concluída + código ausente'
                WHEN task_done = 0 AND exists_at_anchor = 1
                    THEN 'aberta + código existe'
                WHEN task_done = 0 AND exists_at_anchor = 0
                    THEN 'aberta + código ausente'
            END AS state,
            CASE
                WHEN task_done = 1 AND exists_at_anchor = 1 THEN 1
                WHEN task_done = 1 AND exists_at_anchor = 0 THEN 2
                WHEN task_done = 0 AND exists_at_anchor = 1 THEN 3
                WHEN task_done = 0 AND exists_at_anchor = 0 THEN 4
            END AS state_order
        FROM eligible_refs
    )
    SELECT
        task_done,
        state,
        COUNT(*) AS n_references,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_of_eligible,
        ROUND(
            100.0 * COUNT(*) /
            SUM(COUNT(*)) OVER (PARTITION BY task_done),
            2
        ) AS pct_within_task_status
    FROM states
    GROUP BY task_done, state, state_order
    ORDER BY state_order
    """
).df()

matrix

,task_done,state,n_references,pct_of_eligible,pct_within_task_status
0,1,concluída + código existe,297822,72.10,89.69
1,1,concluída + código ausente,34227,8.29,10.31
2,0,aberta + código existe,63261,15.32,78.09
3,0,aberta + código ausente,17752,4.30,21.91


In [7]:
concentration = duckdb.sql(
    f"""
    WITH eligible_refs AS (
        SELECT
            repo_name,
            task_done,
            exists_at_anchor
        FROM read_parquet('{DATA_PATH.as_posix()}')
        WHERE exists_at_anchor IN (0, 1)
    ),
    per_repo AS (
        SELECT
            repo_name,
            COUNT(*) FILTER (
                WHERE task_done = 1
            ) AS completed_references,
            COUNT(*) FILTER (
                WHERE task_done = 1 AND exists_at_anchor = 0
            ) AS completed_missing_references
        FROM eligible_refs
        GROUP BY repo_name
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                ORDER BY completed_missing_references DESC, repo_name
            ) AS repo_rank
        FROM per_repo
    )
    SELECT
        COUNT(*) FILTER (
            WHERE completed_missing_references > 0
        ) AS repositories_with_completed_missing,
        SUM(completed_missing_references) AS completed_missing_references,
        ROUND(
            100.0 * SUM(completed_missing_references)
            FILTER (WHERE repo_rank = 1)
            / NULLIF(SUM(completed_missing_references), 0),
            2
        ) AS pct_in_top_1_repository,
        ROUND(
            100.0 * SUM(completed_missing_references)
            FILTER (WHERE repo_rank <= 10)
            / NULLIF(SUM(completed_missing_references), 0),
            2
        ) AS pct_in_top_10_repositories,
        ROUND(
            100.0 * SUM(completed_missing_references)
            FILTER (WHERE repo_rank <= 50)
            / NULLIF(SUM(completed_missing_references), 0),
            2
        ) AS pct_in_top_50_repositories
    FROM ranked
    """
).df()

concentration

,repositories_with_completed_missing,completed_missing_references,pct_in_top_1_repository,pct_in_top_10_repositories,pct_in_top_50_repositories
0,3746,34227.0,5.49,12.98,24.21


In [8]:
ARTIFACT_PATH = PROJECT_ROOT / "data/raw/openspec_artifact_files.parquet"

assert ARTIFACT_PATH.exists(), f"Arquivo não encontrado: {ARTIFACT_PATH}"

artifact_schema = duckdb.sql(
    f"DESCRIBE SELECT * FROM read_parquet('{ARTIFACT_PATH.as_posix()}')"
).df()

artifact_schema

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,repo_name,VARCHAR,YES,None,None,None
2,file_path,VARCHAR,YES,None,None,None
3,kind,VARCHAR,YES,None,None,None
4,content,VARCHAR,YES,None,None,None
5,content_sha,VARCHAR,YES,None,None,None
6,n_bytes,BIGINT,YES,None,None,None
7,n_tasks,BIGINT,YES,None,None,None
8,n_tasks_done,BIGINT,YES,None,None,None
9,n_code_refs,BIGINT,YES,None,None,None


In [9]:
join_coverage = duckdb.sql(
    f"""
    WITH reference_artifacts AS (
        SELECT
            repo_name,
            artifact_path,
            COUNT(*) AS n_references
        FROM read_parquet('{DATA_PATH.as_posix()}')
        GROUP BY repo_name, artifact_path
    ),
    artifact_keys AS (
        SELECT
            repo_name,
            file_path,
            COUNT(*) AS n_artifact_rows
        FROM read_parquet('{ARTIFACT_PATH.as_posix()}')
        GROUP BY repo_name, file_path
    ),
    joined AS (
        SELECT
            r.repo_name,
            r.artifact_path,
            r.n_references,
            a.n_artifact_rows
        FROM reference_artifacts AS r
        LEFT JOIN artifact_keys AS a
            ON r.repo_name = a.repo_name
           AND r.artifact_path = a.file_path
    )
    SELECT
        COUNT(*) AS distinct_reference_artifacts,
        COUNT(*) FILTER (
            WHERE n_artifact_rows IS NOT NULL
        ) AS matched_artifacts,
        COUNT(*) FILTER (
            WHERE n_artifact_rows IS NULL
        ) AS unmatched_artifacts,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE n_artifact_rows IS NOT NULL
            ) / COUNT(*),
            2
        ) AS artifact_match_pct,
        SUM(n_references) AS reference_rows,
        SUM(n_references) FILTER (
            WHERE n_artifact_rows IS NOT NULL
        ) AS matched_reference_rows,
        ROUND(
            100.0 * SUM(n_references) FILTER (
                WHERE n_artifact_rows IS NOT NULL
            ) / SUM(n_references),
            2
        ) AS reference_match_pct,
        COUNT(*) FILTER (
            WHERE n_artifact_rows > 1
        ) AS duplicate_artifact_keys
    FROM joined
    """
).df()

join_coverage

,distinct_reference_artifacts,matched_artifacts,unmatched_artifacts,artifact_match_pct,reference_rows,matched_reference_rows,reference_match_pct,duplicate_artifact_keys
0,62472,62472,0,100.0,435401.0,435401.0,100.0,0


In [10]:
predictor_quality = duckdb.sql(
    f"""
    WITH reference_artifacts AS (
        SELECT
            repo_name,
            artifact_path,
            COUNT(*) AS n_references
        FROM read_parquet('{DATA_PATH.as_posix()}')
        GROUP BY repo_name, artifact_path
    )
    SELECT
        a.kind,
        COUNT(*) AS n_artifacts,
        SUM(r.n_references) AS n_references,
        COUNT(*) FILTER (
            WHERE a.n_tasks IS NULL OR a.n_tasks = 0
        ) AS artifacts_without_tasks,
        COUNT(*) FILTER (
            WHERE a.n_tasks_done IS NULL
        ) AS artifacts_with_missing_done_count,
        COUNT(*) FILTER (
            WHERE a.n_code_refs IS NULL OR a.n_code_refs = 0
        ) AS artifacts_without_code_refs,
        MIN(a.n_tasks) AS min_tasks,
        MAX(a.n_tasks) AS max_tasks,
        MIN(a.n_code_refs) AS min_code_refs,
        MAX(a.n_code_refs) AS max_code_refs
    FROM reference_artifacts AS r
    INNER JOIN read_parquet('{ARTIFACT_PATH.as_posix()}') AS a
        ON r.repo_name = a.repo_name
       AND r.artifact_path = a.file_path
    GROUP BY a.kind
    ORDER BY n_references DESC
    """
).df()

predictor_quality

,kind,n_artifacts,n_references,artifacts_without_tasks,artifacts_with_missing_done_count,artifacts_without_code_refs,min_tasks,max_tasks,min_code_refs,max_code_refs
0,tasks.md,62472,435401.0,728,0,0,0,964,1,60


In [11]:
cohort_flow = duckdb.sql(
    f"""
    WITH joined AS (
        SELECT
            r.id,
            r.task_done,
            r.exists_at_anchor,
            a.n_tasks,
            a.n_tasks_done,
            a.n_code_refs,
            a.n_bytes
        FROM read_parquet('{DATA_PATH.as_posix()}') AS r
        INNER JOIN read_parquet('{ARTIFACT_PATH.as_posix()}') AS a
            ON r.repo_name = a.repo_name
           AND r.artifact_path = a.file_path
    )
    SELECT
        COUNT(*) AS all_references,
        COUNT(*) FILTER (
            WHERE exists_at_anchor IN (0, 1)
        ) AS binary_resolved_references,
        COUNT(*) FILTER (
            WHERE exists_at_anchor = 2
        ) AS unexpected_state_2,
        COUNT(*) FILTER (
            WHERE exists_at_anchor IS NULL
        ) AS unresolved_references,
        COUNT(*) FILTER (
            WHERE n_tasks = 0
        ) AS references_in_zero_task_artifacts,
        COUNT(*) FILTER (
            WHERE n_tasks_done > n_tasks
        ) AS invalid_task_counts,
        COUNT(*) FILTER (
            WHERE exists_at_anchor IN (0, 1)
              AND n_tasks > 0
        ) AS structural_analysis_references,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE exists_at_anchor IN (0, 1)
                  AND n_tasks > 0
            ) / COUNT(*),
            2
        ) AS structural_analysis_pct
    FROM joined
    """
).df()

cohort_flow

,all_references,binary_resolved_references,unexpected_state_2,unresolved_references,references_in_zero_task_artifacts,invalid_task_counts,structural_analysis_references,structural_analysis_pct
0,435401,413062,130,22209,8093,0,408190,93.75


In [12]:
predictor_summary = duckdb.sql(
    f"""
    WITH referenced_artifacts AS (
        SELECT DISTINCT
            repo_name,
            artifact_path
        FROM read_parquet('{DATA_PATH.as_posix()}')
        WHERE exists_at_anchor IN (0, 1)
    ),
    eligible_artifacts AS (
        SELECT
            a.repo_name,
            a.file_path,
            a.n_tasks,
            a.n_tasks_done,
            a.n_code_refs,
            a.n_bytes,
            a.n_tasks_done::DOUBLE / a.n_tasks AS completion_rate,
            a.n_code_refs::DOUBLE / a.n_tasks AS reference_density
        FROM read_parquet('{ARTIFACT_PATH.as_posix()}') AS a
        INNER JOIN referenced_artifacts AS r
            ON a.repo_name = r.repo_name
           AND a.file_path = r.artifact_path
        WHERE a.n_tasks > 0
    )
    SELECT
        COUNT(*) AS n_artifacts,
        MIN(n_tasks) AS min_tasks,
        MEDIAN(n_tasks) AS median_tasks,
        QUANTILE_CONT(n_tasks, 0.95) AS p95_tasks,
        MAX(n_tasks) AS max_tasks,
        MIN(n_code_refs) AS min_code_refs,
        MEDIAN(n_code_refs) AS median_code_refs,
        QUANTILE_CONT(n_code_refs, 0.95) AS p95_code_refs,
        MAX(n_code_refs) AS max_code_refs,
        ROUND(MEDIAN(completion_rate), 3) AS median_completion_rate,
        ROUND(QUANTILE_CONT(completion_rate, 0.25), 3) AS p25_completion_rate,
        ROUND(QUANTILE_CONT(completion_rate, 0.75), 3) AS p75_completion_rate,
        ROUND(MEDIAN(reference_density), 3) AS median_reference_density,
        ROUND(QUANTILE_CONT(reference_density, 0.95), 3) AS p95_reference_density,
        ROUND(MEDIAN(n_bytes), 0) AS median_bytes,
        QUANTILE_CONT(n_bytes, 0.95) AS p95_bytes,
        MAX(n_bytes) AS max_bytes
    FROM eligible_artifacts
    """
).df()

predictor_summary

,n_artifacts,min_tasks,median_tasks,p95_tasks,max_tasks,min_code_refs,median_code_refs,p95_code_refs,max_code_refs,median_completion_rate,p25_completion_rate,p75_completion_rate,median_reference_density,p95_reference_density,median_bytes,p95_bytes,max_bytes
0,60278,1,20.0,71.0,964,1,4.0,22.0,60,1.0,0.97,1.0,0.225,0.8,2957.0,12161.0,849322


In [13]:
completion_groups = duckdb.sql(
    f"""
    WITH eligible_refs AS (
        SELECT
            r.repo_name,
            r.artifact_path,
            r.exists_at_anchor,
            a.n_tasks,
            a.n_tasks_done
        FROM read_parquet('{DATA_PATH.as_posix()}') AS r
        INNER JOIN read_parquet('{ARTIFACT_PATH.as_posix()}') AS a
            ON r.repo_name = a.repo_name
           AND r.artifact_path = a.file_path
        WHERE r.exists_at_anchor IN (0, 1)
          AND a.n_tasks > 0
    ),
    artifact_outcomes AS (
        SELECT
            repo_name,
            artifact_path,
            MAX(n_tasks) AS n_tasks,
            MAX(n_tasks_done) AS n_tasks_done,
            COUNT(*) AS n_references,
            SUM(
                CASE
                    WHEN exists_at_anchor = 0 THEN 1
                    ELSE 0
                END
            ) AS n_missing
        FROM eligible_refs
        GROUP BY repo_name, artifact_path
    )
    SELECT
        CASE
            WHEN n_tasks_done = 0 THEN 'nenhuma tarefa concluída'
            WHEN n_tasks_done = n_tasks THEN 'todas as tarefas concluídas'
            ELSE 'conclusão parcial'
        END AS completion_group,
        COUNT(*) AS n_artifacts,
        SUM(n_references) AS n_references,
        SUM(n_missing) AS n_missing,
        ROUND(
            100.0 * SUM(n_missing) / SUM(n_references),
            2
        ) AS missing_reference_pct
    FROM artifact_outcomes
    GROUP BY completion_group
    ORDER BY
        CASE completion_group
            WHEN 'nenhuma tarefa concluída' THEN 1
            WHEN 'conclusão parcial' THEN 2
            WHEN 'todas as tarefas concluídas' THEN 3
        END
    """
).df()

completion_groups

,completion_group,n_artifacts,n_references,n_missing,missing_reference_pct
0,nenhuma tarefa concluída,4565,32804.0,9096.0,27.73
1,conclusão parcial,11301,98398.0,15684.0,15.94
2,todas as tarefas concluídas,44412,276988.0,26434.0,9.54


In [14]:
task_count_quartiles = duckdb.sql(
    f"""
    WITH eligible_refs AS (
        SELECT
            r.repo_name,
            r.artifact_path,
            r.exists_at_anchor,
            a.n_tasks
        FROM read_parquet('{DATA_PATH.as_posix()}') AS r
        INNER JOIN read_parquet('{ARTIFACT_PATH.as_posix()}') AS a
            ON r.repo_name = a.repo_name
           AND r.artifact_path = a.file_path
        WHERE r.exists_at_anchor IN (0, 1)
          AND a.n_tasks > 0
    ),
    artifact_outcomes AS (
        SELECT
            repo_name,
            artifact_path,
            MAX(n_tasks) AS n_tasks,
            COUNT(*) AS n_references,
            SUM(
                CASE
                    WHEN exists_at_anchor = 0 THEN 1
                    ELSE 0
                END
            ) AS n_missing
        FROM eligible_refs
        GROUP BY repo_name, artifact_path
    ),
    binned_artifacts AS (
        SELECT
            *,
            NTILE(4) OVER (ORDER BY n_tasks) AS task_count_quartile
        FROM artifact_outcomes
    )
    SELECT
        task_count_quartile,
        MIN(n_tasks) AS min_tasks,
        MAX(n_tasks) AS max_tasks,
        COUNT(*) AS n_artifacts,
        SUM(n_references) AS n_references,
        ROUND(
            100.0 * SUM(n_missing) / SUM(n_references),
            2
        ) AS pooled_missing_pct,
        ROUND(
            100.0 * AVG(
                n_missing::DOUBLE / n_references
            ),
            2
        ) AS mean_artifact_missing_pct
    FROM binned_artifacts
    GROUP BY task_count_quartile
    ORDER BY task_count_quartile
    """
).df()

task_count_quartiles

,task_count_quartile,min_tasks,max_tasks,n_artifacts,n_references,pooled_missing_pct,mean_artifact_missing_pct
0,1,1,12,15070,44537.0,7.40,7.77
1,2,12,20,15070,71458.0,8.75,9.43
2,3,20,33,15069,105343.0,10.83,11.42
3,4,33,964,15069,186852.0,16.19,16.81


In [15]:
reference_density_quartiles = duckdb.sql(
    f"""
    WITH eligible_refs AS (
        SELECT
            r.repo_name,
            r.artifact_path,
            r.exists_at_anchor,
            a.n_tasks,
            a.n_code_refs
        FROM read_parquet('{DATA_PATH.as_posix()}') AS r
        INNER JOIN read_parquet('{ARTIFACT_PATH.as_posix()}') AS a
            ON r.repo_name = a.repo_name
           AND r.artifact_path = a.file_path
        WHERE r.exists_at_anchor IN (0, 1)
          AND a.n_tasks > 0
    ),
    artifact_outcomes AS (
        SELECT
            repo_name,
            artifact_path,
            MAX(n_tasks) AS n_tasks,
            MAX(n_code_refs) AS n_code_refs,
            MAX(n_code_refs)::DOUBLE / MAX(n_tasks) AS reference_density,
            COUNT(*) AS n_references,
            SUM(
                CASE
                    WHEN exists_at_anchor = 0 THEN 1
                    ELSE 0
                END
            ) AS n_missing
        FROM eligible_refs
        GROUP BY repo_name, artifact_path
    ),
    binned_artifacts AS (
        SELECT
            *,
            NTILE(4) OVER (
                ORDER BY reference_density
            ) AS density_quartile
        FROM artifact_outcomes
    )
    SELECT
        density_quartile,
        ROUND(MIN(reference_density), 3) AS min_density,
        ROUND(MAX(reference_density), 3) AS max_density,
        COUNT(*) AS n_artifacts,
        SUM(n_references) AS n_references,
        ROUND(
            100.0 * SUM(n_missing) / SUM(n_references),
            2
        ) AS pooled_missing_pct,
        ROUND(
            100.0 * AVG(
                n_missing::DOUBLE / n_references
            ),
            2
        ) AS mean_artifact_missing_pct
    FROM binned_artifacts
    GROUP BY density_quartile
    ORDER BY density_quartile
    """
).df()

reference_density_quartiles

,density_quartile,min_density,max_density,n_artifacts,n_references,pooled_missing_pct,mean_artifact_missing_pct
0,1,0.003,0.111,15070,32747.0,17.50,16.12
1,2,0.111,0.225,15070,67466.0,13.68,10.87
2,3,0.225,0.400,15069,110349.0,11.72,9.13
3,4,0.400,12.000,15069,197628.0,11.80,9.30


In [16]:
analysis_artifacts = duckdb.sql(
    f"""
    WITH joined AS (
        SELECT
            r.repo_name,
            r.artifact_path,
            r.task_done,
            r.exists_at_anchor,
            a.n_tasks,
            a.n_tasks_done,
            a.n_code_refs,
            a.n_bytes
        FROM read_parquet('{DATA_PATH.as_posix()}') AS r
        INNER JOIN read_parquet('{ARTIFACT_PATH.as_posix()}') AS a
            ON r.repo_name = a.repo_name
           AND r.artifact_path = a.file_path
        WHERE r.exists_at_anchor IN (0, 1)
          AND a.n_tasks > 0
    )
    SELECT
        repo_name,
        artifact_path,
        MAX(n_tasks) AS n_tasks,
        MAX(n_tasks_done) AS n_tasks_done,
        MAX(n_code_refs) AS n_code_refs,
        MAX(n_bytes) AS n_bytes,
        MAX(n_tasks_done)::DOUBLE / MAX(n_tasks) AS completion_rate,
        MAX(n_code_refs)::DOUBLE / MAX(n_tasks) AS reference_density,
        COUNT(*) AS n_references,
        SUM(
            CASE WHEN exists_at_anchor = 0 THEN 1 ELSE 0 END
        ) AS n_missing,
        SUM(
            CASE WHEN task_done = 1 THEN 1 ELSE 0 END
        ) AS n_completed_references,
        SUM(
            CASE
                WHEN task_done = 1 AND exists_at_anchor = 0 THEN 1
                ELSE 0
            END
        ) AS n_completed_missing
    FROM joined
    GROUP BY repo_name, artifact_path
    """
).df()

ANALYSIS_PATH = PROJECT_ROOT / "data/interim/pilot_artifact_outcomes.parquet"

analysis_artifacts.to_parquet(ANALYSIS_PATH, index=False)

print(f"Artefatos: {len(analysis_artifacts):,}")
print(f"Arquivo salvo: {ANALYSIS_PATH}")
analysis_artifacts.head()

Artefatos: 60,278
Arquivo salvo: /Users/pedroodorize/.codex/.chatgpt-projects/g-p-6abaf5bf87e481919dc29fec1c4d1433/msr2027-specmine/data/interim/pilot_artifact_outcomes.parquet


,repo_name,artifact_path,n_tasks,n_tasks_done,n_code_refs,n_bytes,completion_rate,reference_density,n_references,n_missing,n_completed_references,n_completed_missing
0,0serna/dotfiles,openspec/changes/archive/2026-04-07-migrate-bu...,12,12,1,1153,1.0,0.083333,1,1.0,1.0,1.0
1,0serna/dotfiles,openspec/changes/archive/2026-04-10-track-glob...,6,6,1,942,1.0,0.166667,1,1.0,1.0,1.0
2,0serna/dotfiles,openspec/changes/archive/2026-04-24-migrate-bi...,24,24,1,2023,1.0,0.041667,1,0.0,1.0,0.0
3,0serna/dotfiles,openspec/changes/archive/2026-06-23-add-cloudf...,16,16,4,2358,1.0,0.250000,4,0.0,4.0,0.0
4,0serna/dotfiles,openspec/changes/archive/2026-06-23-improve-we...,26,26,3,3157,1.0,0.115385,3,0.0,3.0,0.0


In [17]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

model_df = analysis_artifacts.copy()

model_df["log_tasks"] = np.log1p(model_df["n_tasks"])
model_df["log_bytes"] = np.log1p(model_df["n_bytes"])

model_df["reference_density_z"] = (
    model_df["reference_density"] - model_df["reference_density"].mean()
) / model_df["reference_density"].std()

model_df["completion_group"] = np.select(
    [
        model_df["n_tasks_done"] == 0,
        model_df["n_tasks_done"] == model_df["n_tasks"],
    ],
    [
        "none_complete",
        "all_complete",
    ],
    default="partial_complete",
)

model_df = model_df.dropna(
    subset=[
        "repo_name",
        "n_missing",
        "n_references",
        "log_tasks",
        "log_bytes",
        "reference_density_z",
    ]
).copy()

X = pd.DataFrame(
    {
        "intercept": 1.0,
        "log_tasks": model_df["log_tasks"],
        "reference_density_z": model_df["reference_density_z"],
        "log_bytes": model_df["log_bytes"],
        "partial_complete": (
            model_df["completion_group"] == "partial_complete"
        ).astype(float),
        "none_complete": (
            model_df["completion_group"] == "none_complete"
        ).astype(float),
    }
)

endog = np.column_stack(
    [
        model_df["n_missing"].astype(int),
        (model_df["n_references"] - model_df["n_missing"]).astype(int),
    ]
)

glm = sm.GLM(
    endog,
    X,
    family=sm.families.Binomial(),
)

glm_result = glm.fit(
    cov_type="cluster",
    cov_kwds={"groups": model_df["repo_name"]},
)

confidence_intervals = glm_result.conf_int()

model_effects = pd.DataFrame(
    {
        "predictor": glm_result.params.index,
        "odds_ratio": np.exp(glm_result.params),
        "ci_low": np.exp(confidence_intervals[0]),
        "ci_high": np.exp(confidence_intervals[1]),
        "p_value": glm_result.pvalues,
    }
).round(3)

print(f"Artefatos no modelo: {len(model_df):,}")
print(f"Repositórios no modelo: {model_df['repo_name'].nunique():,}")

model_effects

Artefatos no modelo: 60,278
Repositórios no modelo: 6,869


,predictor,odds_ratio,ci_low,ci_high,p_value
intercept,intercept,0.019,0.009,0.041,0.000
log_tasks,log_tasks,1.538,1.369,1.727,0.000
reference_density_z,reference_density_z,1.035,1.001,1.070,0.047
log_bytes,log_bytes,1.028,0.911,1.159,0.657
partial_complete,partial_complete,1.463,1.314,1.629,0.000
none_complete,none_complete,3.191,2.701,3.770,0.000


In [18]:
completed_model_df = model_df.loc[
    model_df["n_completed_references"] > 0
].copy()

X_completed = pd.DataFrame(
    {
        "intercept": 1.0,
        "log_tasks": completed_model_df["log_tasks"],
        "reference_density_z": completed_model_df["reference_density_z"],
        "log_bytes": completed_model_df["log_bytes"],
        "partial_complete": (
            completed_model_df["completion_group"] == "partial_complete"
        ).astype(float),
    }
)

endog_completed = np.column_stack(
    [
        completed_model_df["n_completed_missing"].astype(int),
        (
            completed_model_df["n_completed_references"]
            - completed_model_df["n_completed_missing"]
        ).astype(int),
    ]
)

completed_glm = sm.GLM(
    endog_completed,
    X_completed,
    family=sm.families.Binomial(),
)

completed_glm_result = completed_glm.fit(
    cov_type="cluster",
    cov_kwds={"groups": completed_model_df["repo_name"]},
)

completed_ci = completed_glm_result.conf_int()

completed_effects = pd.DataFrame(
    {
        "predictor": completed_glm_result.params.index,
        "odds_ratio": np.exp(completed_glm_result.params),
        "ci_low": np.exp(completed_ci[0]),
        "ci_high": np.exp(completed_ci[1]),
        "p_value": completed_glm_result.pvalues,
    }
).round(3)

print(f"Artefatos no modelo: {len(completed_model_df):,}")
print(
    "Referências concluídas no modelo: "
    f"{completed_model_df['n_completed_references'].sum():,}"
)

completed_effects

Artefatos no modelo: 53,128
Referências concluídas no modelo: 332,049.0


,predictor,odds_ratio,ci_low,ci_high,p_value
intercept,intercept,0.026,0.011,0.064,0.000
log_tasks,log_tasks,1.510,1.304,1.749,0.000
reference_density_z,reference_density_z,1.032,0.983,1.083,0.206
log_bytes,log_bytes,0.993,0.859,1.147,0.922
partial_complete,partial_complete,1.252,1.128,1.390,0.000


In [20]:
repo_sizes = (
    completed_model_df.groupby("repo_name")["n_completed_references"]
    .sum()
    .sort_values(ascending=False)
)

top_50_repos = set(repo_sizes.head(50).index)

sensitivity_df = completed_model_df.loc[
    ~completed_model_df["repo_name"].isin(top_50_repos)
].copy()

X_sensitivity = pd.DataFrame(
    {
        "intercept": 1.0,
        "log_tasks": sensitivity_df["log_tasks"],
        "reference_density_z": sensitivity_df["reference_density_z"],
        "log_bytes": sensitivity_df["log_bytes"],
        "partial_complete": (
            sensitivity_df["completion_group"] == "partial_complete"
        ).astype(float),
    }
)

endog_sensitivity = np.column_stack(
    [
        sensitivity_df["n_completed_missing"].astype(int),
        (
            sensitivity_df["n_completed_references"]
            - sensitivity_df["n_completed_missing"]
        ).astype(int),
    ]
)

sensitivity_result = sm.GLM(
    endog_sensitivity,
    X_sensitivity,
    family=sm.families.Binomial(),
).fit(
    cov_type="cluster",
    cov_kwds={"groups": sensitivity_df["repo_name"]},
)

sensitivity_ci = sensitivity_result.conf_int()

pd.DataFrame(
    {
        "predictor": sensitivity_result.params.index,
        "odds_ratio": np.exp(sensitivity_result.params),
        "ci_low": np.exp(sensitivity_ci[0]),
        "ci_high": np.exp(sensitivity_ci[1]),
        "p_value": sensitivity_result.pvalues,
    }
).round(3)

,predictor,odds_ratio,ci_low,ci_high,p_value
intercept,intercept,0.018,0.008,0.044,0.000
log_tasks,log_tasks,1.480,1.259,1.739,0.000
reference_density_z,reference_density_z,1.018,0.964,1.076,0.510
log_bytes,log_bytes,1.044,0.887,1.230,0.604
partial_complete,partial_complete,1.294,1.183,1.416,0.000


In [21]:
VALIDATION_SEED = 20260928
VALIDATION_PATH = PROJECT_ROOT / "data/interim/validation_candidates_pilot.csv"

validation_candidates = duckdb.sql(
    f"""
    WITH classified AS (
        SELECT
            id,
            repo_name,
            artifact_path,
            code_path,
            task_done,
            anchor_sha,
            exists_at_anchor,
            resolved_path,
            CASE
                WHEN task_done = 1 AND exists_at_anchor = 0
                    THEN 'concluída + ausente'
                WHEN task_done = 1 AND exists_at_anchor = 1
                    THEN 'concluída + existe'
                WHEN task_done = 0 AND exists_at_anchor = 0
                    THEN 'aberta + ausente'
            END AS validation_group
        FROM read_parquet('{DATA_PATH.as_posix()}')
        WHERE
            (task_done = 1 AND exists_at_anchor IN (0, 1))
            OR (task_done = 0 AND exists_at_anchor = 0)
    ),
    ranked AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY validation_group
                ORDER BY hash(id, {VALIDATION_SEED})
            ) AS sample_rank
        FROM classified
        WHERE validation_group IS NOT NULL
    )
    SELECT
        id,
        repo_name,
        artifact_path,
        code_path,
        task_done,
        anchor_sha,
        exists_at_anchor,
        resolved_path,
        validation_group,
        sample_rank
    FROM ranked
    WHERE sample_rank <= 20
    ORDER BY validation_group, sample_rank
    """
).df()

validation_candidates.to_csv(VALIDATION_PATH, index=False)

print(f"Candidatos selecionados: {len(validation_candidates)}")
print(f"Arquivo salvo: {VALIDATION_PATH}")
validation_candidates.groupby("validation_group").size()

Candidatos selecionados: 60
Arquivo salvo: /Users/pedroodorize/.codex/.chatgpt-projects/g-p-6abaf5bf87e481919dc29fec1c4d1433/msr2027-specmine/data/interim/validation_candidates_pilot.csv


validation_group
aberta + ausente       20
concluída + ausente    20
concluída + existe     20
dtype: int64

In [22]:
from urllib.parse import quote
import csv

AUDIT_PATH = PROJECT_ROOT / "data/interim/validation_workbook_pilot.csv"

audit_con = duckdb.connect()
audit_con.register("validation_candidates", validation_candidates)

audit_context = audit_con.execute(
    f"""
    SELECT
        c.*,
        a.content AS artifact_content
    FROM validation_candidates AS c
    JOIN read_parquet('{ARTIFACT_PATH.as_posix()}') AS a
        ON c.repo_name = a.repo_name
        AND c.artifact_path = a.file_path
    """
).df()

def excerpt_around_reference(content, code_path, before=250, after=450):
    if content is None:
        return ""

    position = content.find(code_path)

    if position == -1:
        position = content.find(code_path.removeprefix("./"))

    if position == -1:
        return "[referência não localizada literalmente no texto]"

    excerpt = content[max(0, position - before): position + len(code_path) + after]
    return " ".join(excerpt.split())

def github_blob_url(repo_name, sha, path):
    safe_path = quote(str(path), safe="/")
    return f"https://github.com/{repo_name}/blob/{sha}/{safe_path}"

workbook = audit_context.assign(
    artifact_url=lambda df: [
        github_blob_url(repo, sha, path)
        for repo, sha, path in zip(
            df["repo_name"], df["anchor_sha"], df["artifact_path"]
        )
    ],
    code_url=lambda df: [
        github_blob_url(repo, sha, path)
        for repo, sha, path in zip(
            df["repo_name"], df["anchor_sha"], df["code_path"]
        )
    ],
    reference_excerpt=lambda df: [
        excerpt_around_reference(content, code_path)
        for content, code_path in zip(df["artifact_content"], df["code_path"])
    ],
    audit_status="pending",
    manual_classification="",
    evidence_note="",
).loc[
    :,
    [
        "id",
        "validation_group",
        "repo_name",
        "anchor_sha",
        "artifact_path",
        "code_path",
        "task_done",
        "exists_at_anchor",
        "artifact_url",
        "code_url",
        "reference_excerpt",
        "audit_status",
        "manual_classification",
        "evidence_note",
    ],
]

workbook.to_csv(AUDIT_PATH, index=False, quoting=csv.QUOTE_ALL)

print(f"Planilha de auditoria salva: {AUDIT_PATH}")
workbook[
    [
        "validation_group",
        "repo_name",
        "artifact_path",
        "code_path",
        "reference_excerpt",
    ]
].head(3)

Planilha de auditoria salva: /Users/pedroodorize/.codex/.chatgpt-projects/g-p-6abaf5bf87e481919dc29fec1c4d1433/msr2027-specmine/data/interim/validation_workbook_pilot.csv


,validation_group,repo_name,artifact_path,code_path,reference_excerpt
0,concluída + existe,lewisoepwqi/scientific_nini,openspec/changes/archive/2026-03-26-add-eviden...,tools/registry.py,中挂载 EvidenceCollector 实例 ## 3. Skill 契约集成 - [x...
1,concluída + ausente,liverty-music/specification,openspec/changes/archive/2026-03-11-improve-on...,my-app.ts,"dia query to disable pulse animation, keep sta..."
2,concluída + ausente,liverty-music/specification,openspec/changes/archive/2026-03-15-extract-sh...,src/components/svg-icon/svg-icon.css,svg-icon>` component - [x] 1.1 Create `src/com...


In [23]:
workbook = (
    workbook
    .merge(
        validation_candidates[["id", "sample_rank"]],
        on="id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(["validation_group", "sample_rank"])
)

workbook.to_csv(AUDIT_PATH, index=False, quoting=csv.QUOTE_ALL)

pilot_audit = (
    workbook.loc[
        workbook["validation_group"] == "concluída + ausente"
    ]
    .sort_values("sample_rank")
    .head(5)
)

pd.set_option("display.max_colwidth", None)

pilot_audit[
    [
        "id",
        "sample_rank",
        "repo_name",
        "artifact_path",
        "code_path",
        "artifact_url",
        "code_url",
        "reference_excerpt",
    ]
]

,id,sample_rank,repo_name,artifact_path,code_path,artifact_url,code_url,reference_excerpt
40,69114,1,clawmasons/mason,openspec/changes/archive/2026-03-15-agent-packages/tasks.md,packages/cli/src/materializer/mcp-agent.ts,https://github.com/clawmasons/mason/blob/f019e52a01045e200fbfe74e3cc79e58edb67d5a/openspec/changes/archive/2026-03-15-agent-packages/tasks.md,https://github.com/clawmasons/mason/blob/f019e52a01045e200fbfe74e3cc79e58edb67d5a/packages/cli/src/materializer/mcp-agent.ts,"ts` to `packages/pi-coding-agent/tests/materializer.test.ts`, update imports - [x] 3.5 Add `packages/pi-coding-agent` to monorepo workspace config ## 4. Add Materializer to Existing `@clawmasons/mcp-agent` Package - [x] 4.1 Move materializer from `packages/cli/src/materializer/mcp-agent.ts` to `packages/mcp-agent/src/materializer.ts`, update imports to use `@clawmasons/agent-sdk` - [x] 4.2 Create default `AgentPackage` export in `packages/mcp-agent/src/index.ts` with name `""mcp-agent""`, aliases `[""mcp""]`, ACP command, and runtime config - [x] 4.3 Move tests from `packages/cli/tests/materializer/mcp-agent.test.ts` to `packages/mcp-agent/tests/materializer.test.ts`, update imports - [x] 4.4 Add `@clawmasons/agent-sdk` as dependency"
24,42703,2,BlackBeltTechnology/pi-agent-dashboard,openspec/changes/archive/2026-04-27-fix-flow-step-agent-tracking/tasks.md,agent-card.ts,https://github.com/BlackBeltTechnology/pi-agent-dashboard/blob/672bf366e73ed248cd6f0da8be419d9cfdac5cf5/openspec/changes/archive/2026-04-27-fix-flow-step-agent-tracking/tasks.md,https://github.com/BlackBeltTechnology/pi-agent-dashboard/blob/672bf366e73ed248cd6f0da8be419d9cfdac5cf5/agent-card.ts,"call at line ~809) - [x] 1.3 Add `onAgentStarted`/`onAgentComplete` calls to `spawnForkDecisionAgent` in `flow-execution.ts` (wrap around `spawnAgent()` call at line ~585) ## 2. Pi-flows: Fix TUI card rendering with renderBox() - [x] 2.1 Refactor `agent-card.ts` to use `renderBox()` from `box-renderer.ts` instead of manually building `┌─┐ │ └─┘` borders — SKIPPED: card borders work correctly, the actual bug was grid truncation. renderBox refactor deferred as cosmetic. - [x] 2.2 Update `grid-component.ts` to compute row height from actual rendered card line count instead of hardcoded `CARD_HEIGHT=8` ## 3. Dashboard: Add stepType to FlowAgentState - [x] 3.1 Add optional `stepType?: string` field to `Fl"
54,31364,3,AynSiddiqui/Playground,openspec/changes/archive/2026-05-24-cpp-memory-visualizer/tasks.md,D3.js,https://github.com/AynSiddiqui/Playground/blob/6836895151fe77ca0526b819d05000fa39b7d95d/openspec/changes/archive/2026-05-24-cpp-memory-visualizer/tasks.md,https://github.com/AynSiddiqui/Playground/blob/6836895151fe77ca0526b819d05000fa39b7d95d/D3.js,"with Vite - [x] 4.2 Set up the main split-pane layout - [x] 4.3 Integrate Monaco Editor with C++ syntax highlighting - [x] 4.4 Implement WebSocket client logic to connect to the backend ## 5. Memory Visualization UI - [x] 5.1 Set up React Flow or D3.js canvas in the visualization pane - [x] 5.2 Create visual nodes representing primitive variables, structs, and STL abstractions based on the snapshot JSON `isStl` flags - [x] 5.3 Implement pointer rendering (edges connecting nodes) ## 6. Playback & State Management - [x] 6.1 Implement the frontend cache array to store historical snapshots - [x] 6.2 Build the timeline playback controls (Next, Previous, Reset) - [x] 6.3 Wire the ""Next"" button to"
49,308872,4,Vsssel/StackView,openspec/changes/task-status-update-endpoint/tasks.md,TaskDetailPage.tsx,https://github.com/Vsssel/StackView/blob/8fa93da1f2076a2b66ec4e07423be693c90dd4bb/openspec/changes/task-status-update-endpoint/tasks.md,https://github.com/Vsssel/StackView/blob/8fa93da1f2076a2b66ec4e07423be693c90dd4bb/TaskDetailPage.tsx,ify optimistic UI and cache update - [ ] 3.5 Test: Permission denied scenario (if testable) and error rollback ## 4. Task Detail Components - [x] 4.1 Update `TaskDetailSlideOver.tsx` status dropdown to use `useUpdateTaskS